# The Solver API

`model.solve(...)` is enough for almost every use case — see [Solving a model](solving.ipynb). When you need finer control — adjusting the native solver model before it runs, obtaining the `Result` object directly, or bracketing model transformations by hand — you can build a `Solver` in one step and run it in another.

In [ ]:
import pandas as pd

from linopy import Model
from linopy.solvers import Solver


def build_model():
    m = Model()
    x = m.add_variables(lower=0, name="x")
    y = m.add_variables(lower=0, name="y")
    m.add_constraints(3 * x + 7 * y >= 10)
    m.add_constraints(5 * x + 2 * y >= 3)
    m.add_objective(x + 2 * y)
    return m

## Building the solver

Build the solver with `Solver.from_name`, then run it with `.solve()`. The two steps are separate, so you can inspect or adjust the solver between them:

In [ ]:
m = build_model()
solver = Solver.from_name("highs", m, io_api="direct", options={"output_flag": False})
solver

`solver.solver_model` is the native solver handle — here a `highspy.Highs` instance. You could tweak it directly before running:

In [ ]:
solver.solver_model

In [ ]:
result = solver.solve()
result

`Result` carries the status, solution, solver name, and report. Writing it back into the `Model`, combining numeric values with labels and coordinates, is a separate call:

In [ ]:
m.assign_result(result)
m.objective.value

In [ ]:
m.solution.to_pandas()

## Model transformations

**Model transformations live on the `Model`, not the `Solver`.** A `Solver` only declares which features it supports and raises during `_build()` if it can't handle the model it's been handed; it never mutates the model. The transformations currently exposed:

| Transformation | Methods on `Model` | Reversible? |
|---|---|---|
| SOS reformulation (rewrite SOS constraints as Big-M binary + linear) | `model.apply_sos_reformulation()` / `model.undo_sos_reformulation()` | yes |
| Drop zero-coefficient terms | `model.constraints.sanitize_zeros()` | one-way |
| Replace ±inf bounds in constraints | `model.constraints.sanitize_infinities()` | one-way |

`model.solve(reformulate_sos=True, sanitize_zeros=True, sanitize_infinities=True)` is a convenience that **brackets** these around the one-shot solve (and undoes the SOS reformulation afterwards). The two-step `Solver` API does **not** do this for you — when you go through `Solver.from_name(...).solve()`, you call the transformations yourself first, and use `try`/`finally` to keep the model in a known state if the solve raises:

In [ ]:
m = Model()
i = pd.Index([0, 1, 2], name="i")
x = m.add_variables(lower=0, upper=1, coords=[i], name="x")
m.add_sos_constraints(x, sos_type=1, sos_dim="i")
m.add_objective(x.sum(), sense="max")

m.constraints.sanitize_zeros()
m.constraints.sanitize_infinities()

m.apply_sos_reformulation()
try:
    solver = Solver.from_name(
        "highs", m, io_api="direct", options={"output_flag": False}
    )
    result = solver.solve()
    m.assign_result(result)
finally:
    m.undo_sos_reformulation()  # restore original SOS form on the Model

list(m.variables.sos)